# IMDAA Regridding Notebook

This notebook demonstrates how to regrid IMDAA reanalysis data from its native resolution (0.12°) to any user-specified target resolution using bilinear interpolation.

## Overview

The IMDAA (Indian Meteorological Department Analysis and Assimilation) reanalysis dataset is available at a native spatial resolution of 0.12° over the Indian domain (5°N–40°N, 65°E–100°E). This notebook provides a reproducible workflow to:

1. Load the original IMDAA NetCDF files
2. Select the Indian domain
3. Define a target output grid at any desired resolution
4. Perform bilinear regridding using `xesmf`
5. Visualise and save the regridded output

The BharatBench dataset (available at https://www.kaggle.com/datasets/maslab/bharatbench) was prepared at 1.08° resolution (32×32 grid points) using this approach. Users who require a different resolution can download the original IMDAA data from https://rds.ncmrwf.gov.in/datasets and use this script to regrid it accordingly.

## Requirements

- `xarray`
- `xesmf`
- `numpy`
- `matplotlib`
- `cartopy`
- `cftime`

Install with:
```bash
pip install xarray xesmf numpy matplotlib cartopy cftime
```


## 1. Imports

In [ ]:
import numpy as np
import xarray as xr
import xesmf as xe
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cftime

%matplotlib inline

## 2. User Configuration

Set the paths and target resolution here before running the notebook.

In [ ]:
# -----------------------------------------------------------------------
# USER CONFIGURATION
# -----------------------------------------------------------------------

# Path to the original IMDAA NetCDF files (supports wildcards for multiple files)
# Download from: https://rds.ncmrwf.gov.in/datasets
INPUT_PATH = '/path/to/imdaa/data/*.nc'

# Name of the variable to regrid (e.g., 'HGT_prl', 'TMP_2m', 'APCP_sfc')
VARIABLE_NAME = 'HGT_prl'

# Target output resolution in degrees
# BharatBench uses 1.08 degrees (32x32 grid points over the Indian domain)
# Change this to any desired resolution (e.g., 0.25, 0.5, 1.0)
TARGET_RESOLUTION_DEG = 1.08

# Domain bounds (Indian subcontinent)
LAT_MIN, LAT_MAX = 5.03, 38.65
LON_MIN, LON_MAX = 65.03, 98.65

# Path for the regridded output file
OUTPUT_PATH = '/path/to/output/IMDAA_regridded.nc'

# Regridding method: 'bilinear' (recommended), 'conservative', or 'nearest_s2d'
REGRID_METHOD = 'bilinear'

# -----------------------------------------------------------------------

## 3. Load and Inspect the Input Data

In [ ]:
# Load all input files and combine along the time dimension
ds = xr.open_mfdataset(INPUT_PATH, combine='by_coords')
print('Input dataset summary:')
print(ds)

## 4. Select the Target Variable and Spatial Domain

In [ ]:
# Select the spatial domain (Indian subcontinent)
ds = ds.sel(
    latitude=slice(LAT_MIN, LAT_MAX),
    longitude=slice(LON_MIN, LON_MAX)
)

# Extract the variable of interest
da = ds[VARIABLE_NAME]
print(f'Variable "{VARIABLE_NAME}" selected:')
print(da)

## 5. Visualise a Sample of the Input Data (Optional)

In [ ]:
# Plot a single time step to inspect the input data
# Adjust the 'isel' parameters as needed for your variable
fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()}, figsize=(8, 6))

if 'plevel' in da.dims:
    da.isel(time=0, plevel=7).plot.pcolormesh(ax=ax)
else:
    da.isel(time=0).plot.pcolormesh(ax=ax)

ax.coastlines()
ax.set_title(f'Input data: {VARIABLE_NAME} (native resolution)')
plt.tight_layout()
plt.show()

## 6. Define the Target Grid and Regrid

In [ ]:
# Define the output grid at the desired target resolution
lat_out = np.arange(LAT_MIN + 0.01, LAT_MAX, TARGET_RESOLUTION_DEG)
lon_out = np.arange(LON_MIN + 0.01, LON_MAX, TARGET_RESOLUTION_DEG)

ds_out = xr.Dataset(
    data_vars=None,
    coords={
        'latitude':  (['latitude'],  lat_out, {'units': 'degrees_north'}),
        'longitude': (['longitude'], lon_out, {'units': 'degrees_east'}),
    }
)

print(f'Target grid: {len(lat_out)} x {len(lon_out)} points '
      f'at {TARGET_RESOLUTION_DEG}° resolution')
print(ds_out)

In [ ]:
# Build the regridder
regridder = xe.Regridder(ds, ds_out, REGRID_METHOD)
print(regridder)

# Apply the regridder to the selected variable
da_out = regridder(da, keep_attrs=True)
print('\nRegridded output:')
print(da_out)

## 7. Visualise the Regridded Output (Optional)

In [ ]:
# Plot the same time step after regridding for comparison
fig, ax = plt.subplots(subplot_kw={'projection': ccrs.PlateCarree()}, figsize=(8, 6))

if 'plevel' in da_out.dims:
    da_out.isel(time=0, plevel=7).plot.pcolormesh(ax=ax)
else:
    da_out.isel(time=0).plot.pcolormesh(ax=ax)

ax.coastlines()
ax.set_title(f'Regridded data: {VARIABLE_NAME} ({TARGET_RESOLUTION_DEG}°)')
plt.tight_layout()
plt.show()

## 8. Save the Regridded Data to NetCDF

In [ ]:
# Convert DataArray back to Dataset and save
ds_regridded = da_out.to_dataset()
ds_regridded.to_netcdf(path=OUTPUT_PATH, mode='w')
print(f'Regridded data saved to: {OUTPUT_PATH}')

## 9. Verify the Saved Output

In [ ]:
# Reload and inspect the saved file to confirm it is correct
ds_check = xr.open_dataset(OUTPUT_PATH)
print('Saved dataset summary:')
print(ds_check)

## Notes

- The `xesmf` regridder builds interpolation weights on the first call. For large datasets, this may take some time but the weights are reusable across multiple variables on the same grid.
- The `bilinear` method is recommended for smooth fields (temperature, geopotential height, wind). For precipitation or other fields with sharp gradients, `conservative` interpolation may be more appropriate.
- The BharatBench dataset (1.08°, 32×32 grid) was produced using the `bilinear` method, consistent with the approach described in the accompanying manuscript.
- For further details on the dataset and baseline benchmarks, see: https://github.com/MASLABnitrkl/BharatBench
